# Lab 06: Streaming and Batch

**Goal:** Learn the three ways to run a chain -- invoke, stream, and batch.

What you'll learn:
- `.invoke()` -- run once, get complete result
- `.stream()` -- get tokens as they're generated (real-time)
- `.batch()` -- process multiple inputs at once

In [ ]:
import time
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm = ChatOllama(model="llama3.2:1b")

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant. Keep responses to 2-3 sentences."),
    ("human", "{question}"),
])

chain = prompt | llm | StrOutputParser()

## Step 1: `.invoke()` -- Wait for Complete Response

You've been using this. It waits until the full response is ready.

In [ ]:
start = time.time()
result = chain.invoke({"question": "What is cloud computing?"})
elapsed = time.time() - start

print(result)
print(f"(Completed in {elapsed:.1f}s -- you waited for the entire response)")

## Step 2: `.stream()` -- Real-time Token Output

In a chat UI, users don't want to wait 5 seconds staring at a blank screen.
`.stream()` yields tokens as the LLM generates them -- feels much faster.

In [ ]:
start = time.time()

for chunk in chain.stream({"question": "What is cloud computing?"}):
    print(chunk, end="", flush=True)  # flush=True ensures immediate display

elapsed = time.time() - start
print(f"\n(Streamed in {elapsed:.1f}s -- text appeared as it was generated)")

## Step 3: `.batch()` -- Process Multiple Inputs

When you need answers to many questions, batch is more efficient
than calling invoke in a loop.

In [ ]:
start = time.time()

questions = [
    {"question": "What is Flask?"},
    {"question": "What is FastAPI?"},
    {"question": "What is Terraform?"},
]

results = chain.batch(questions)

elapsed = time.time() - start
for q, r in zip(questions, results):
    print(f"Q: {q['question']}")
    print(f"A: {r}\n")

print(f"(Processed {len(questions)} questions in {elapsed:.1f}s)")

## Step 4: Compare -- Invoke Loop vs Batch

In [ ]:
# Sequential invoke
start = time.time()
for q in questions:
    chain.invoke(q)
invoke_time = time.time() - start

# Batch
start = time.time()
chain.batch(questions)
batch_time = time.time() - start

print(f"Sequential .invoke() x{len(questions)}: {invoke_time:.1f}s")
print(f"Single .batch() call:        {batch_time:.1f}s")
print(f"Batch is {'faster' if batch_time < invoke_time else 'similar speed'} on this model")

## TODO 1: Build a Streaming Code Explainer

Create a chain that explains a code snippet line by line,
and use `.stream()` to show the explanation appearing in real-time.

- Create a `code_explain_prompt` with variables `{language}` and `{code}`
- Build a chain using `code_explain_prompt | llm | StrOutputParser()`
- Use `.stream()` to explain a short Python function

In [ ]:
# code_explain_prompt = ChatPromptTemplate.from_messages([
#     ("system", "You are a {language} expert. Explain the code line by line in plain English. Be concise."),
#     ("human", "Explain this code:\n```\n{code}\n```"),
# ])
# code_explain_chain = code_explain_prompt | llm | StrOutputParser()
#
# code_snippet = """def fibonacci(n):
#     a, b = 0, 1
#     for _ in range(n):
#         a, b = b, a + b
#     return a"""
#
# print("Explaining code (streaming):\n")
# for chunk in code_explain_chain.stream({"language": "Python", "code": code_snippet}):
#     print(chunk, end="", flush=True)
# print()

## TODO 2: Batch Process — Analyze Error Messages

Use `.batch()` to analyze 5 common error messages in parallel.

- Create a list of 5 error dicts with realistic error messages
- Use `chain.batch()` (or create a new diagnosis chain) to process them
- Print the diagnosis for each error

In [ ]:
# error_prompt = ChatPromptTemplate.from_messages([
#     ("system", "You are a debugging expert. Diagnose the error and suggest a fix in 2 sentences."),
#     ("human", "{question}"),
# ])
# error_chain = error_prompt | llm | StrOutputParser()
#
# errors = [
#     {"question": "Diagnose: ModuleNotFoundError: No module named 'pandas'"},
#     {"question": "Diagnose: ConnectionRefusedError: [Errno 111] Connection refused on port 5432"},
#     {"question": "Diagnose: PermissionError: [Errno 13] Permission denied: '/etc/config.yaml'"},
#     {"question": "Diagnose: TypeError: unsupported operand type(s) for +: 'int' and 'str'"},
#     {"question": "Diagnose: RecursionError: maximum recursion depth exceeded"},
# ]
#
# start = time.time()
# results = error_chain.batch(errors)
# elapsed = time.time() - start
#
# for err, diagnosis in zip(errors, results):
#     print(f"Error: {err['question'].replace('Diagnose: ', '')}")
#     print(f"Fix:   {diagnosis}\n")
#
# print(f"(Batch-diagnosed {len(errors)} errors in {elapsed:.1f}s)")

## Key Takeaways

- `.invoke()` -- complete result, good for scripts/APIs
- `.stream()` -- real-time tokens, good for chat UIs
- `.batch()` -- multiple inputs, good for bulk processing
- All three work on ANY LCEL chain -- no extra code needed